# 🎯 [K12_M06] 절차를 단계별 형식으로 안내하기
### 학생 실습 워크북 & 프롬프트 설계 가이드 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K12/k12_m06_student.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 일반행정·부대관리 (Level 1) / K12 행정 업무 FAQ 챗봇 설계 |
| **미션** | 미션 06. 절차를 단계별 형식으로 안내하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 생성형 AI 시스템 프롬프트 엔지니어링 및 팩트체크 검증 워크북 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.



In [ ]:
# ============================================================================
# [환경 설정 및 헬퍼 함수] 마크다운 시각화 렌더러 및 가상 데이터셋 로더
# ============================================================================
from IPython.display import display, Markdown
import json
import os

def load_dataset(mission_num):
    """미션 번호(1~10)에 해당하는 가상 데이터셋 JSON을 로드하여 반환합니다."""
    filename = f"k12_m{mission_num:02d}_dataset.json"
    for base in [".", "uploads/K12", "../uploads/K12"]:
        filepath = os.path.join(base, filename)
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                return json.load(f)
    print(f"⚠️ 데이터셋 파일을 찾을 수 없습니다: {filename}")
    return None

def show_dataset_summary(mission_num):
    """해당 미션의 데이터셋 주요 항목을 요약 출력합니다."""
    data = load_dataset(mission_num)
    if data:
        display(Markdown(f"**[데이터셋 로드 성공]** `{data.get('title', '')}` ({data.get('mission', '')})"))
        keys = [k for k in data.keys() if k not in ["project", "mission", "title"]]
        print(f"포함 항목: {', '.join(keys)}")

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 AI 답변 및 검증 결과를 시각적으로 렌더링하는 헬퍼 함수"""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 AI 생성 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ 워크북 시각화 헬퍼 함수 및 가상 데이터셋 로더(k12_m01~m10) 준비 완료!")



---

## 🎯 미션 06. 절차를 단계별 형식으로 안내하기
**소요시간**: 약 1.5시간

![미션 6 6단 단계별 행정 안내 표준 출력 서식 제어 및 Few-shot 모범 답변 최적화 다이어그램](/uploads/common/K12_m06_format_fewshot.png)

### 📋 미션 개요 및 배경
장병들이 복잡한 행정 절차를 한눈에 파악하고 즉각 실행할 수 있도록 일관된 단계별 출력 서식을 규격화합니다. '질문 요약 → 단계별 절차(1·2·3단계) → 필요 서류 → 기한 및 주의사항 → 담당 부서 → 근거 번호' 템플릿과 Few-shot(예시)을 시스템 프롬프트에 정의합니다. 반말, 축약어, 오타가 포함된 불완전한 사용자 질의에도 일관된 서식과 가독성을 유지하는지 스트레스 테스트를 수행하여 출력 제어 역량을 완성합니다.

### 🎯 미션 학습목표
'질문 요약 → 절차 1·2·3단계 → 필요 서류 → 기한·주의 → 담당 부서 → 근거 번호' 형식을 정하고 예시 1개를 넣습니다. 반말·줄임말·오타 질문에도 형식이 유지되는지 확인합니다. **이를 통해 Few-shot 예시와 템플릿 제약을 활용하여 비정형 사용자 입력 환경에서도 규격화된 정보 가독성을 보장하는 출력 서식 제어 역량을 익힙니다.**

### 📑 실습 가이드 및 세부 단계 (프롬프트 & 검증 포인트)
#### 🔹 Step 1. 단계별 답변 형식을 정한다.
#### 이 단계의 역할

장병들이 복잡한 행정 절차를 한눈에 파악하고 즉각 실행할 수 있도록 일관된 단계별 출력 서식을 규격화합니다.

#### 해야 할 일

- [질문 요약 → 단계별 절차(1·2·3단계) → 필요 서류 → 기한 및 주의사항 → 담당 부서 → 근거 번호]의 표준 출력 템플릿을 정의합니다.

- 모바일 인트라넷 화면에서도 줄바꿈과 불릿이 깔끔하게 표시되도록 마크다운 서식을 설계합니다.

- AI에게 가독성을 극대화할 수 있는 서식 구조를 자문합니다.

#### 💡 AI 프롬프트 질의 예시

> **[표준 단계별 출력 포맷 설계]:** "장병들이 모바일이나 PC 화면에서 복잡한 행정 절차를 즉각 이해할 수 있도록 [질문 요약 / 단계별 절차 / 필요 서류 / 신청 기한 및 유의사항 / 담당 부서 및 문의 / 근거 FAQ] 구조의 표준 출력 서식 템플릿을 마크다운 형식으로 제안해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **필수 항목 망라:** 절차, 서류, 기한, 부서, 근거 5대 필수 정보가 모두 템플릿 항목에 포함되었는가?
- **시각적 가독성:** 계층 구조(헤더, 불릿)가 직관적인가?

#### 핵심 확인 지표

> 단계별 행정 안내 표준 출력 템플릿

#### 학습자가 제출해야 하는 내용

- 출력 서식 템플릿 정의서

#### 🔹 Step 2. 예시 1개를 넣는다.
#### 이 단계의 역할

정의된 표준 출력 서식에 맞춰 모범 응답 예시(Few-shot) 1건을 작성하여 시스템 프롬프트에 주입합니다.

#### 해야 할 일

- 대표 행정 업무 1건(예: 군 복무확인서/경력증명서 발급)을 선택합니다.

- 앞서 정의한 템플릿의 모든 필드를 FAQ 원문에 맞게 완벽히 작성한 모범 응답 1건을 작성합니다.

- 시스템 프롬프트의 [답변 예시] 블록에 해당 Few-shot 예시를 탑재합니다.

#### 💡 AI 프롬프트 질의 예시

> **[Few-shot 모범 응답 예시 작성]:** "앞서 설계한 표준 출력 서식에 맞춰 '군 경력증명서 발급 절차'에 대한 완벽한 모범 응답 예시 1건을 작성해 줘. 시스템 지시문 내 [출력 형식 및 예시] 섹션에 포함할 수 있도록 정제해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **Few-shot 무결성:** 예시 내용이 FAQ 표 원문 데이터(서류, 기한, 담당부서)와 100% 일치하는가?
- **포맷 일치:** 설계한 템플릿의 항목명과 배치 순서를 정확히 따랐는가?

#### 핵심 확인 지표

> Few-shot 예시가 포함된 고도화 시스템 프롬프트

#### 학습자가 제출해야 하는 내용

- 모범 응답 예시가 탑재된 프롬프트 전문

#### 🔹 Step 3. 반말·줄임말·오타 질문으로 시험한다.
#### 이 단계의 역할

반말("휴가 어케 신청함?"), 줄임말("경증 어서 뗌?"), 오타("비번 초기화 어떠케") 등 거친 실제 질문을 입력하여 규격화된 서식이 유지되는지 스트레스 테스트를 수행합니다.

#### 해야 할 일

- 신병이나 급한 장병들이 작성할 법한 비표준 질문 3종(반말형, 축약형, 오타형)을 준비합니다.

- 고도화된 챗봇에 질의하여 AI가 사용자 말투에 휘말리지 않고 정중한 톤과 표준 서식을 유지하는지 확인합니다.

- 응답 로그를 기록하고 서식 붕괴나 어조 왜곡 여부를 평가합니다.

#### 💡 AI 프롬프트 질의 예시

> <p>**[비표준 질의 스트레스 테스트 실행]:** "{Few-shot 포함 챗봇 프롬프트}

[장병 질문]: 포상휴가 짤릴까봐 그런데 신청 언제까정 해야댐? 서류 머필요?"</p>

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **출력 제어 견고성:** 반말 질문에도 정중한 경어체와 6단계 표준 마크다운 서식을 완벽히 지켰는가?
- **의도 파악 정확도:** 줄임말과 오타 속에서도 핵심 행정 질문(포상휴가 신청 기한 및 서류)을 정확히 인식했는가?

#### 핵심 확인 지표

> 비표준 질문 3종 스트레스 테스트 결과서

#### 학습자가 제출해야 하는 내용

- 스트레스 테스트 질의 응답 로그

#### 🔹 Step 4. 심화 도전 과제 및 최적화
#### 이 단계의 역할

단계별 절차 중 장병이 스스로 처리할 온라인 단계와 간부/행정반 대면 승인 단계를 시각적으로 구분하는 아이콘 서식을 추가합니다.

#### 해야 할 일

- [온라인 신청: 💻], [행정반 대면 제출: 📝], [지휘관 결재: 🎖️] 등 시각적 아이콘 규칙을 구상합니다.

- 챗봇이 절차 단계별로 적절한 아이콘을 붙여 출력하도록 서식을 미세 튜닝합니다.

#### 💡 AI 프롬프트 질의 예시

> **[절차 구분 아이콘 서식 확장]:** "절차의 성격(온라인 전산 입력 vs 오프라인 대면 서류 제출)을 직관적으로 구분할 수 있는 아이콘 표기 가이드를 출력 형식에 추가해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **가독성 개선:** 아이콘이 과도하게 남발되지 않고 핵심 행동 구분에 도움이 되는가?

#### 핵심 확인 지표

> 아이콘 서식이 반영된 심화 출력 테스트 메모

#### 학습자가 제출해야 하는 내용

- 심화 도전 과제 메모

### 📦 미션 필수 학습자료 및 팁
**미션 수행 필수 학습자료**

- **6단 표준 단계별 행정 답변 포맷 명세서:** [📌 질문 요약 / 📝 단계별 절차 / 📁 필요 서류 / ⏰ 기한 및 유의사항 / 📞 담당 부서 / 🔍 근거 번호] 템플릿
- **Few-shot 모범 질의응답 예시문 (군 경력증명서 발급 샘플):** AI의 출력 서식, 글머리 기호, 줄바꿈을 일관되게 제어하는 모범 샷(Shot)
- **[학습 가이드] Few-shot 프롬프팅과 비정형 입력 스트레스 테스트:** 사용자 질문이 반말, 줄임말, 오타여도 규격화된 정중한 답변을 강제하는 프롬프트 견고성(Robustness) 검증법

**실습 팁:** Few-shot 예시에 들어가는 글머리 기호(1단계, 2단계 등)와 섹션 제목을 그대로 유지하도록 지시문에 한 줄 명시하면 답변의 포맷 일관성이 비약적으로 높아집니다.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [선택] 출력 형식 지정 — 6단 표준 단계별 서식을 규격화하고, 비표준 질문(반말/축약/오타)에도 서식 붕괴 없이 일관되게 응답하도록 제어했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 6단 표준 단계별 출력 템플릿 및 Few-shot 모범 응답 완성

- 비표준 질문 3종(반말, 축약어, 오타) 스트레스 테스트 수행

- 어조 유지 및 서식 준수율 100% 확보

#### 채점자 판정 메모

> "야 포상휴가 신청 어케 하냐?"라는 반말 질문에 대해 AI가 존댓말과 6단계 마크다운 서식을 그대로 지켜 답변했는지 확인한다.



In [ ]:
# [미션 06.] 학생 실습 - 프롬프트 및 결과 기록 셀
# 💡 [데이터셋 확인 팁]: 아래 주석을 해제하여 본 미션의 가상 데이터셋 요약을 확인할 수 있습니다.
# show_dataset_summary(6)

# TODO 1: 위 실습 가이드와 AI 질의 예시를 참고하여 작성한 프롬프트를 입력하세요.
mission_06_prompt = """
# 여기에 본인이 설계한 프롬프트를 작성하세요.
"""

# TODO 2: 생성형 AI가 출력한 답변 및 원문 대조(Fact-check) 검증 내역을 기록하세요.
mission_06_result = """
# 여기에 AI 생성 결과 및 팩트체크 대조표/산출물을 마크다운 서식으로 기록하세요.
"""

# 렌더러 실행 (작성한 내용을 시각적으로 확인합니다)
show_result("미션 06. 절차를 단계별 형식으로 안내하기", mission_06_prompt, mission_06_result)

